In [1]:
from torchsummary import summary
import os
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import sys

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
from src.utils import *
from src.training import *
from src.visualization import *
from configs import config
from src.checkpoints  import TrainingState, load_checkpoint, save_checkpoint, AccuracyTracker
from models.Unet import get_model
from src.load_data import *
from src.checkpoints import *
from src.preprocessing import *


In [3]:
device= torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes=2
learning_rate = config.learning_rate
batch_size = config.batch_size
num_epochs = 2
loss_fn = nn.CrossEntropyLoss()
img_height = config.unet_height
img_width  = config.unet_width
weight_decay=1e-4

accuracy_result_name="Unet_qualitative_results"
sub_checkpoint_path="unet_mobilenetv2.pth"
model_name='Unet'

In [4]:
sub_checkpoint_dir=os.path.join("outputs","checkpoints")
sub_dataset_directory=os.path.join("data","datasets")
sub_prediction_folder=os.path.join( "outputs",accuracy_result_name)

In [5]:
checkpoint_dir = os.path.join(project_root,sub_checkpoint_dir)
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir,sub_checkpoint_path )
prediction_folder=os.path.join(project_root,sub_prediction_folder)
dataset_directory=os.path.join(project_root,sub_dataset_directory)

In [6]:
saved_path_train = os.path.join(project_root,sub_prediction_folder,'Predictions','train')
saved_path_validation = os.path.join(project_root,sub_prediction_folder,'Predictions','validation')
saved_path_test = os.path.join(project_root,sub_prediction_folder,'Predictions','test')

# 1.得到所有image的名字，方便后续调用

In [7]:
paths = get_data_paths(dataset_directory)
image_list_train, mask_list_train=get_sorted_image_mask_lists(paths['image_train'], paths['mask_train'])
image_list_validation, mask_list_validation=get_sorted_image_mask_lists(paths['image_validation'], paths['mask_validation'])
image_list_test, mask_list_test=get_sorted_image_mask_lists(paths['image_test'], paths['mask_test'])

In [8]:
train_transform,val_test_transform,train_dataset, validation_dataset, test_dataset = create_datasets(
    image_list_train, mask_list_train,
    image_list_validation, mask_list_validation,
    image_list_test, mask_list_test,
    img_height, img_width
)

# 2.训练数据的批量加载器

In [9]:
train_loader, validation_loader, test_loader = create_dataloaders(
    train_dataset, validation_dataset, test_dataset,
    batch_size=batch_size, num_workers=0, pin_memory=True
)


Testing the train_loader...
Images batch shape: torch.Size([16, 3, 224, 224])
Masks batch shape: torch.Size([16, 224, 224])
Image dtype: torch.float32
Mask dtype: torch.int64


# 3. 加载模型，优化器，查看参数配置

In [10]:
model = get_model(n_classes=num_classes)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [11]:
summary(model, (3, 224, 224))

Layer (type:depth-idx)                             Output Shape              Param #
├─Sequential: 1-1                                  [-1, 16, 112, 112]        --
|    └─Conv2dNormActivation: 2-1                   [-1, 32, 112, 112]        --
|    |    └─Conv2d: 3-1                            [-1, 32, 112, 112]        (864)
|    |    └─BatchNorm2d: 3-2                       [-1, 32, 112, 112]        (64)
|    |    └─ReLU6: 3-3                             [-1, 32, 112, 112]        --
|    └─InvertedResidual: 2-2                       [-1, 16, 112, 112]        --
|    |    └─Sequential: 3-4                        [-1, 16, 112, 112]        (896)
├─Sequential: 1-2                                  [-1, 24, 56, 56]          --
|    └─InvertedResidual: 2-3                       [-1, 24, 56, 56]          --
|    |    └─Sequential: 3-5                        [-1, 24, 56, 56]          (5,136)
|    └─InvertedResidual: 2-4                       [-1, 24, 56, 56]          --
|    |    └─Sequential

Layer (type:depth-idx)                             Output Shape              Param #
├─Sequential: 1-1                                  [-1, 16, 112, 112]        --
|    └─Conv2dNormActivation: 2-1                   [-1, 32, 112, 112]        --
|    |    └─Conv2d: 3-1                            [-1, 32, 112, 112]        (864)
|    |    └─BatchNorm2d: 3-2                       [-1, 32, 112, 112]        (64)
|    |    └─ReLU6: 3-3                             [-1, 32, 112, 112]        --
|    └─InvertedResidual: 2-2                       [-1, 16, 112, 112]        --
|    |    └─Sequential: 3-4                        [-1, 16, 112, 112]        (896)
├─Sequential: 1-2                                  [-1, 24, 56, 56]          --
|    └─InvertedResidual: 2-3                       [-1, 24, 56, 56]          --
|    |    └─Sequential: 3-5                        [-1, 24, 56, 56]          (5,136)
|    └─InvertedResidual: 2-4                       [-1, 24, 56, 56]          --
|    |    └─Sequential

# 4.如果模型已经存在，则调用

In [12]:
state = TrainingState(model=model, optimizer=optimizer)
load_checkpoint(state, checkpoint_path, device)

checkpoint is found: C:\Users\xiaoh\Desktop\3rd-Paper\Segmentation_Pipeline\outputs\checkpoints\unet_mobilenetv2.pth，training is restarted...
the best accuracy from previous training: 0.9626
Checkpoint loaded: from epoch : 20


C:\Users\xiaoh\Desktop\3rd-Paper\Segmentation_Pipeline\src\checkpoints.py:67: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_loca

# 5.开始对模型进行训练

In [13]:
for epoch in range(num_epochs):
    print(f"\n===== Epoch {epoch + 1} / {num_epochs} =====")
    train_fn, eval_fn = get_training_functions(model_name)
    train_loss, train_acc = train_fn(model, train_loader, optimizer, loss_fn, device)
    val_loss, val_acc = eval_fn(model, validation_loader, loss_fn, device)

    state.history['train_loss'].append(train_loss)
    state.history['train_acc'].append(train_acc)
    state.history['val_loss'].append(val_loss)
    state.history['val_acc'].append(val_acc)

    print(f"Epoch {epoch + 1} Summary:")
    print(f"  Training Loss:    {train_loss:.4f} | Training Accuracy:    {train_acc:.4f}")
    print(f"  Validation Loss:  {val_loss:.4f} | Validation Accuracy:  {val_acc:.4f}")
    state.last_val_acc=val_acc
    state.epoch+=1

    save_checkpoint(state, checkpoint_path)


===== Epoch 1 / 2 =====


Training:  45%|██████████████████████████▌                                | 31/69 [00:11<00:13,  2.74it/s, loss=0.0998]


KeyboardInterrupt: 

In [ ]:
if not os.path.exists(prediction_folder):
    os.mkdir(prediction_folder)

# 6.查看已经训练好的模型的预测，FN,FP，和原来的图片分列观察

In [14]:
analyze_and_visualize_predictions(train_loader, model,"Unet", device, num_to_show=1)


--- Analyzing Sample 1 ---
Mean IoU: 0.5824
Summation of False Negatives (FN): 1894
Summation of False Positives (FP): 621
Summation of True Positives (TP): 3508
Summation of True Negatives (TN): 44153


[0.5824340029885439]

In [ ]:
generate_merged_images("Unet", model,test_loader,prediction_folder,"test_set")

In [ ]:

results_train = get_iou_list(train_loader, model,model_name)
results_validation = get_iou_list(validation_loader, model,model_name)
results_test = get_iou_list(test_loader, model,model_name)

print('\n================== FINAL RESULTS ==================')
print('Training Set Results      ->', results_train)
print('Validation Set Results    ->', results_validation)
print('Test Set Results          ->', results_test)
print('===================================================')

# 7.得到IOU list并且储存在results_train，results_validation，results_test中

# 8.将IOU的结果存储起来

In [ ]:

save_iou_results_to_file(
    os.path.join(prediction_folder,"Total_IoU_Scores.txt"), 
    results_train, 
    results_validation, 
    results_test
)

# 9. 画IOU scatter

In [ ]:

plot_iou_scatter(
    results_train, 
    results_validation, 
    results_test,
    train_loader,      # Your PyTorch train_dataset instance
    validation_loader, # Your PyTorch validation_dataset instance
    test_loader ,       # Your PyTorch test_dataset instance
    prediction_folder
)

# 10. 根据已有的dict画 Loss 和 Accuracy

In [ ]:

plot_training_history(training_history,prediction_folder)

print("All results have been saved and plotted.")


# 11. 看一下数据量

In [ ]:
# --- Get sizes from the PyTorch Dataset instances ---
size_train = len(train_dataset)
size_validation = len(validation_dataset)
size_test = len(test_dataset)

print('\n--- Dataset Sizes ---')
print(f'Train size: {size_train}, Validation size: {size_validation}, Test size: {size_test}')

# 12. 将dictionary中的结果结果存储到text文件里

In [ ]:
training_history

In [ ]:
save_training_history(training_history, prediction_folder)

# 13. 画图

In [ ]:
plot_training_history_custom(training_history,prediction_folder)

# 在test set里评估结果

In [ ]:
loss_fn

In [ ]:
print("Evaluating final model on the Test Set...")

test_results = final_evaluation(test_loader, model, "Unet",loss_fn, device)

print(f"\n--- Test Set Results ---")
print(f"Time Used For Test Set Evaluation: {test_results['evaluation_time_seconds']:.2f} seconds")
print(f"Test set loss: {test_results['loss']:.4f}")
print(f"Test set accuracy: {test_results['accuracy']:.4f}")
print(f"Test set Mean IoU: {test_results['mean_iou']:.4f}")

# 保存预测的图片结果

In [ ]:

print("\n--- Starting to Save All Model Predictions to Disk ---")

save_all_predictions(
    loader=train_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=saved_path_train,
    name_prefix='train'
)

save_all_predictions(
    loader=validation_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=saved_path_validation,
    name_prefix='validation'
)

save_all_predictions(
    loader=test_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=saved_path_test,
    name_prefix='test'
)

print("\n--- All prediction images have been saved successfully. ---")

# 保存预测的FN,FP的叠加图片

In [ ]:

path_train_superposition = os.path.join(project_root, 'outputs', 'Unet_accuracy_result','Predictions','train_superposition')
path_validation_superposition= os.path.join(project_root, 'outputs', 'Unet_accuracy_result','Predictions','validation_superposition')
path_test_superposition = os.path.join(project_root,  'outputs', 'Unet_accuracy_result','Predictions','test_superposition')
print("\n--- Starting to Save All Error Map Visualizations to Disk ---")


print("\n--- Starting to Save All Error Map Visualizations to Disk ---")

save_error_map_visualizations(
    loader=train_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=path_train_superposition,
    name_prefix='train'
)

save_error_map_visualizations(
    loader=validation_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=path_validation_superposition,
    name_prefix='validation'
)

save_error_map_visualizations(
    loader=test_loader, 
    model=model, 
    model_name="Unet",
    device=device, 
    output_path=path_test_superposition,
    name_prefix='test'
)

print("\n--- All error map images have been saved successfully. ---")

# 分成马氏体，贝氏体，来进行预测

In [ ]:
input_bainite = os.path.join(project_root,'data','datasets','bainite_set','original') 
input_tempered_martensite= os.path.join(project_root,'data','datasets','martensite_set','original') 
output_bainite = os.path.join(project_root, 'outputs', 'Unet_accuracy_result','Predictions','bainite')
output_martensite= os.path.join(project_root, 'outputs', 'Unet_accuracy_result','Predictions','martensite')

predict_and_save_folder(
    model=model,
    model_name="Unet",
    device=device,
    input_folder=input_bainite,
    output_folder=output_bainite,
    input_resize_dim=(224, 224) # Note: (width, height) for cv2/albumentations
)

predict_and_save_folder(
    model=model,
    model_name="Unet",
    device=device,
    input_folder=input_tempered_martensite,
    output_folder=output_martensite,
    input_resize_dim=(224, 224)
)